## Notes

In [ ]:
# can travel for 1/4 sq km in 1hr - 1 flight
# each sq km is 18,000rs

In [ ]:
INDIA_PROJECTED_CRS = "24378"

## Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import openpyxl

import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona

fiona.drvsupport.supported_drivers["KML"] = "rw"

In [ ]:
from gridsample.utils import save_shapefiles
# from gridsample.mapping.plot import create_interactive_map

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
OUTPUT_DATA_DIR = DATA_DIR / "01_processed" / "Goverment Buildings"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

CONSOLIDATED_DATA_PATH = (
    RAW_DATA_DIR / "government_buildings" / "Consolidated_data_09042025.xlsx"
)

In [ ]:
workbook = openpyxl.load_workbook(CONSOLIDATED_DATA_PATH)

## Utilities

In [ ]:
def is_cell_highlighted(cell):
    if cell.fill.fill_type == "solid":
        # Check if the cell has a background color (not white/none)
        if (
            cell.fill.start_color.rgb != "00000000"
            and cell.fill.start_color.rgb != "FFFFFFFF"
        ):
            return True
    return False


def get_highlighted_rows(worksheet):
    highlighted_rows = []

    for row_idx, row in enumerate(worksheet.iter_rows(min_row=2)):  # Skip header row
        if sum(is_cell_highlighted(cell) for cell in row) >= 24:
            highlighted_rows.append(row_idx)

    return highlighted_rows

## Load the secondary data source - East Zone DISCOMs data
For backup latlons.

In [ ]:
df_east_discom_1 = pd.read_excel(
    RAW_DATA_DIR
    / "government_buildings"
    / "East Zone DISCOM data - 10 KW and Above EZ.xlsx",
    sheet_name="10 KW and above(LV1,2,4,5) ",
    header=0,
)

df_east_discom_2 = pd.read_excel(
    RAW_DATA_DIR
    / "government_buildings"
    / "East Zone DISCOM data - 10 KW and Above EZ.xlsx",
    sheet_name="10 KW and above (LV3) SL, WW ",
    header=0,
)

df_east_discom = pd.concat([df_east_discom_1, df_east_discom_2], ignore_index=True)

In [ ]:
df_east_discom

## Seoni
East Zone DISCOM. Drop highlighted rows since they are central gov buildings.

In [ ]:
SAVE_DIR_SEONI = OUTPUT_DATA_DIR / "Seoni"
SAVE_DIR_SEONI.mkdir(parents=True, exist_ok=True)


worksheet_seoni = workbook["Seoni"]

df_seoni = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Seoni", header=0)
df_seoni

In [ ]:
rows_to_drop_seoni = get_highlighted_rows(worksheet_seoni)
df_chhindwara = df_seoni.drop(index=rows_to_drop_seoni).reset_index(drop=True)

# Filter to only building names and lat / long
df_chhindwara.rename(
    columns={
        "Name of institutions/consumer as per billing/connection details": "Name",
        "Latitude of location of consumer's permises": "lat",
        "longitude of location of consumer's permises": "lon",
    },
    inplace=True,
)
print(df_chhindwara[["Name", "lat", "lon"]].isna().sum())

In [ ]:
df_chhindwara.to_csv(SAVE_DIR_SEONI / "cleaned_data.csv", index=False)

## Dhar
West Zone DISCOM. Keep highlighted rows.

In [ ]:
SAVE_DIR_DHAR = OUTPUT_DATA_DIR / "Dhar/"
SAVE_DIR_DHAR.mkdir(parents=True, exist_ok=True)

worksheet_dhar = workbook["Dhar"]

df_dhar = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Dhar", header=0)
df_dhar

In [ ]:
# drop the first row which seems to be meaningless
filtered_df_dhar = df_dhar.drop(index=[0]).reset_index(drop=True)
filtered_df_dhar

In [ ]:
filtered_df_dhar[filtered_df_dhar["latitude & Longitude of premises"].isna()]

In [ ]:
filtered_df_dhar_no_nan = filtered_df_dhar.dropna(
    subset=["Latitude & Longitude of premises"], axis=0
)
filtered_df_dhar_no_nan

In [ ]:
# First split by commas or ". " to get lats
filtered_df_dhar_no_nan["lat"] = filtered_df_dhar_no_nan["Latitude & Longitude of premises"].apply(
    lambda x: str(x).split(",")[0].split(". ")[0]
)

# Then split by "." because some numbers have double decimal points -_-
filtered_df_dhar_no_nan["lat"] = (
    filtered_df_dhar_no_nan["lat"]
    .apply(
        lambda x: x
        if len(x.split(".")) <= 2
        else x.split(".")[0] + "." + x.split(".")[1]
    )
    .astype(float)
)

# INSPECT MANUALLY BECAUSE THERE MAY STILL BE SOME INSANITY
filtered_df_dhar_no_nan["lat"].tolist()

In [ ]:
# Rinse and repeat for lons
filtered_df_dhar_no_nan["lon"] = filtered_df_dhar_no_nan["Latitude & Longitude of premises"].apply(
    lambda x: str(x).split(", ")[-1]
)

# Look for specific cases of outliers and fix:
pattern1 = r"(\d{2}\.\d+,\d{2}\.\d+|\d{2}\.\d+\.\d{2}\.\d+)"
pattern1_matches = filtered_df_dhar_no_nan.loc[
    filtered_df_dhar_no_nan["lon"].str.contains(pattern1, regex=True, na=False), "lon"
]
filtered_df_dhar_no_nan.loc[pattern1_matches.index, "lon"] = pattern1_matches.apply(
    lambda x: str(x).split(",")[-1]
)

pattern2 = r"(\d{2}\.\d+. \d{2}\.\d+|\d{2}\.\d+\.\d{2}\.\d+)"
pattern2_matches = filtered_df_dhar_no_nan.loc[
    filtered_df_dhar_no_nan["lon"].str.contains(pattern2, regex=True, na=False), "lon"
]
filtered_df_dhar_no_nan.loc[pattern2_matches.index, "lon"] = pattern2_matches.apply(
    lambda x: str(x).split(". ")[-1]
)


pattern3 = r"(\d{2}\.\d+\.\d+)"
pattern3_matches = filtered_df_dhar_no_nan.loc[
    filtered_df_dhar_no_nan["lon"].str.contains(pattern3, regex=True, na=False), "lon"
]
filtered_df_dhar_no_nan.loc[pattern3_matches.index, "lon"] = pattern3_matches.apply(
    lambda x: str(x).split(".")[0] + "." + str(x).split(".")[1]
)

pattern4 = r"^[^.\,]*$"
not_nans = filtered_df_dhar_no_nan["lon"] != "nan"
pattern4_matches = filtered_df_dhar_no_nan.loc[
    (not_nans) & (filtered_df_dhar_no_nan["lon"].str.contains(pattern4, regex=True, na=False)),
    "lon",
]
filtered_df_dhar_no_nan.loc[pattern4_matches.index, "lon"] = pattern4_matches.apply(
    lambda x: str(x)[:2] + "." + str(x)[2:]
)

pattern5 = r"(\d{2}\,\d+)"
pattern5_matches = filtered_df_dhar_no_nan.loc[
    filtered_df_dhar_no_nan["lon"].str.contains(pattern5, regex=True, na=False), "lon"
]
filtered_df_dhar_no_nan.loc[pattern5_matches.index, "lon"] = pattern5_matches.apply(
    lambda x: str(x).replace(",", ".")
)


filtered_df_dhar_no_nan.loc[filtered_df_dhar_no_nan["lon"] == "  75. ....3224", "lon"] = "75.3224"

# convert to float
filtered_df_dhar_no_nan["lon"] = filtered_df_dhar_no_nan["lon"].astype(float)

# INSPECT MANUALLY BECAUSE THERE MAY STILL BE SOME INSANITY
filtered_df_dhar_no_nan["lon"].tolist()

In [ ]:
filtered_df_dhar_no_nan["lon"].hist()

In [ ]:
filtered_df_dhar_no_nan.to_csv(SAVE_DIR_DHAR / "cleaned_data.csv", index=False)

## Chhindwara
East Zone DISCOM. Keep highlighted rows.

In [ ]:
SAVE_DIR_CHHINDWARA = OUTPUT_DATA_DIR / "Chhindwara"
SAVE_DIR_CHHINDWARA.mkdir(parents=True, exist_ok=True)

worksheet_chhindwara = workbook["Chhindwara"]

df_chhindwara = pd.read_excel(CONSOLIDATED_DATA_PATH, sheet_name="Chhindwara", header=0)
df_chhindwara

In [ ]:
df_chhindwara["Latitude/Langitude of location of consumer's permises"].replace(
    "0", np.nan, inplace=True
)
df_chhindwara["Latitude/Langitude of location of consumer's permises"].replace(
    0, np.nan, inplace=True
)

In [ ]:
df_chhindwara["lat_original"] = df_chhindwara[
    "Latitude/Langitude of location of consumer's permises"
].apply(lambda x: np.nan if pd.isnull(x) else str(x).split(" ")[0])

# fix random double-decimal issue
df_chhindwara["lat_original"] = df_chhindwara["lat_original"].str.replace('..', '.', regex=False)

# change to float
df_chhindwara["lat_original"] = df_chhindwara["lat_original"].astype(float)

# column to keep track of issues
df_chhindwara["lat_problem"] = "unchecked"
df_chhindwara.loc[df_chhindwara["lat_original"].isna(), "lat_problem"] = "missing"
df_chhindwara.loc[df_chhindwara["lat_original"].between(20, 24), "lat_problem"] = "ok"
df_chhindwara.loc[df_chhindwara["lat_original"].between(2, 3), "lat_problem"] = "2.XX"
df_chhindwara.loc[df_chhindwara["lat_original"].between(1, 2), "lat_problem"] = "1.XX"
df_chhindwara.loc[df_chhindwara["lat_original"] > 50, "lat_problem"] = "> 50"

# replace 2.XX with 22.XX values
df_chhindwara.loc[df_chhindwara["lat_original"].astype(float).between(2, 3), "lat_original"] += 20

df_chhindwara["lat_original"].to_list()

In [ ]:
df_chhindwara["lat_problem"].value_counts()

In [ ]:
df_chhindwara[(df_chhindwara["lat_problem"]!="ok")]

In [ ]:
df_chhindwara["lon_original"] = df_chhindwara[
    "Latitude/Langitude of location of consumer's permises"
].apply(lambda x: np.nan if pd.isnull(x) else str(x).split(" ")[1])
df_chhindwara["lon_original"] = df_chhindwara["lon_original"].astype(float)

# track issues
df_chhindwara["lon_problem"] = "unchecked"
df_chhindwara.loc[df_chhindwara["lon_original"].isna(), "lon_problem"] = "missing"
df_chhindwara.loc[df_chhindwara["lon_original"] < 70, "lon_problem"] = "< 70"
df_chhindwara.loc[df_chhindwara["lon_original"] > 100, "lon_problem"] = "> 100"
df_chhindwara.loc[df_chhindwara["lon_original"].between(70, 100), "lon_problem"] = "ok"
df_chhindwara["lon_original"].to_list()

In [ ]:
df_chhindwara["lon_problem"].value_counts()

In [ ]:
# add the lat lon columns from the discoms data with prefix discom_ to the filtered data. The columns to merge on are "IVRS/Consumer Code as per billing details" on the filtered data and "consumer_no" on the discom data
df_chhindwara = df_chhindwara.merge(
    df_east_discom[["consumer_no", "consumer_latitude", "consumer_longitude"]],
    left_on="IVRS/Consumer Code as per billing details",
    right_on="consumer_no",
    how="left",
)
df_chhindwara.rename(
    columns={
        "IVRS/Consumer Code as per billing details": "discom_consumer_no",
        "consumer_latitude": "lat_discom",
        "consumer_longitude": "lon_discom",
    },
    inplace=True,
)
print(df_chhindwara[["discom_consumer_no", "lat_discom", "lon_discom"]].isna().sum())

In [ ]:
# make new columns called "lat" and "lon" that are the lat and lon columns from the filtered data if they are "ok", otherwise they are the lat and lon columns from the discom data
df_chhindwara["lat"] = df_chhindwara.apply(
    lambda row: row["lat_original"]
    if row["lat_problem"] == "ok" or row["lat_problem"] == "2.XX"
    else row["lat_discom"],
    axis=1,
)
df_chhindwara["lon"] = df_chhindwara.apply(
    lambda row: row["lon_original"]
    if row["lon_problem"] == "ok"
    else row["lon_discom"],
    axis=1,
)

In [ ]:
df_chhindwara["lat"].hist()

In [ ]:
df_chhindwara["lon"].hist()

In [ ]:
df_chhindwara

In [ ]:
bad_df_chhindwara = df_chhindwara[
    ~(
        df_chhindwara["lat_problem"].isin(["ok", "2.XX"])
        & df_chhindwara["lon_problem"].isin(["ok"])
    )
]
bad_df_chhindwara

In [ ]:
bad_df_chhindwara[["lat", "lon"]].dropna()
# 6 of the 12 rescued!

In [ ]:
print("Bad buildings: ", len(bad_df_chhindwara), "of", len(df_chhindwara), "total buildings")
print("Rescued buildings: ", len(bad_df_chhindwara[["lat", "lon"]].dropna()))

In [ ]:
df_chhindwara.dropna(subset=["lat", "lon"])

In [ ]:
df_chhindwara_no_nan = df_chhindwara.dropna(
    subset=["lat", "lon"], axis=0
).reset_index(drop=True)

In [ ]:
df_chhindwara_no_nan.to_csv(SAVE_DIR_CHHINDWARA / "cleaned_data.csv", index=False)

### Plot and check

In [ ]:
gdf_chhindwara = gpd.GeoDataFrame(
    df_chhindwara,
    geometry=gpd.points_from_xy(
        df_chhindwara["lon"], df_chhindwara["lat"]
    ),
    crs=INDIA_PROJECTED_CRS,
)
gdf_chhindwara = gdf_chhindwara.to_crs(epsg=4326)

In [ ]:
gdf_chhindwara_discom = gpd.GeoDataFrame(
    df_chhindwara,
    geometry=gpd.points_from_xy(
        df_chhindwara["lon_discom"], df_chhindwara["lat_discom"]
    ),
    crs=INDIA_PROJECTED_CRS,
)
gdf_chhindwara_discom = gdf_chhindwara_discom.to_crs(epsg=4326)

In [ ]:
ax = gdf_chhindwara.plot(
    color="blue",
    marker="o",
    markersize=5,
    label="Improved",
    figsize=(10, 10),
)
gdf_chhindwara_discom.plot(
    color="green",
    marker="o",
    markersize=2,
    label="DISCOM",
    ax=ax,
)
plt.legend()
plt.title("Chhindwara Buildings")
